In [15]:
# Install numpy in EMR cluster
sc.install_pypi_package("numpy")
import numpy as np

from pyspark.ml.recommendation import ALS
from pyspark.sql import functions as F

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 2.3.2 requires tzdata>=2022.7, which is not installed.
matplotlib 3.9.4 requires contourpy>=1.0.1, which is not installed.
matplotlib 3.9.4 requires cycler>=0.10, which is not installed.
matplotlib 3.9.4 requires fonttools>=4.22.0, which is not installed.
matplotlib 3.9.4 requires importlib-resources>=3.2.0; python_version < "3.10", which is not installed.
matplotlib 3.9.4 requires kiwisolver>=1.3.1, which is not installed.
matplotlib 3.9.4 requires pillow>=8, which is not installed.
pandas 2.3.2 requires python-dateutil>=2.8.2, but you have python-dateutil 2.8.1 which is incompatible.

In [16]:
# get the ml-10M100K folder path from s3 bucket
path = 's3://jonesh-test/ml-10M100K/'

# read ratings.dat from the s3 bucket location with delimeter '::' to Spark DataFrame with header
# inferSchema is set to true to convert any numerical values into integer/float
ratings = (spark.read
          .option('delimiter', '::')
          .option('inferSchema', 'true')
          .csv(path + 'ratings.dat')
          .toDF('UserID', 'MovieID', 'Rating', 'Timestamp'))

# read movies.dat from the s3 bucket location with delimeter '::' to Spark DataFrame with header
# inferSchema is set to true to convert any numerical values into integer/float
movies = (spark.read
         .option('delimiter', '::')
         .option('inferSchema', 'true')
         .csv(path + 'movies.dat')
         .toDF('MovieID', 'Title', 'Genres'))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
# ALS learns hidden patterns (latent features) by converting users and ratings into vectors
# predict the missing ratings using dot product
als = ALS(
    rank=50,  # number of latent features (balances complexity and generalization for the MovieLens 10M dataset)
    maxIter=10,  # ALS usually converges quicker (default value)
    regParam=0.1, # L2 regularization strength (default value)
    userCol='UserID',
    itemCol='MovieID',
    ratingCol='Rating',
    coldStartStrategy='drop' # some users or movies appear only in the test set, not the training set 
    # removes invalid predictions (NaN) so evaluation metrics like RMSE work correctly
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
# Train the model
model = als.fit(ratings)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
movie_factors = model.itemFactors # get the latent factors
movie_factors.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+--------------------+
| id|            features|
+---+--------------------+
| 10|[0.13005322, -0.0...|
| 20|[0.13414788, 0.16...|
| 30|[0.379316, -0.180...|
| 40|[0.23622712, 0.05...|
| 50|[0.26777998, -0.1...|
+---+--------------------+
only showing top 5 rows

In [20]:
# get Toy Story ALS embedding vector as python list
toy_story_vec = movie_factors.filter('id = 1').select('features').first()['features']
toy_story_vec

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[0.3150031566619873, -0.21948567032814026, -0.1790868192911148, 0.21488389372825623, -0.47186562418937683, -0.08558786660432816, -0.09112725406885147, 0.072154700756073, 0.2768399715423584, 0.3999905586242676, -0.3012489676475525, 0.4461914896965027, 0.08083394169807434, 0.0019795196130871773, -0.20846426486968994, -0.0017408025451004505, 0.3032161593437195, 0.4313260018825531, -0.48546427488327026, 0.17604510486125946, 0.10194650292396545, 0.030576638877391815, 0.07829190045595169, 0.09839745610952377, 0.0017936719814315438, 0.040128812193870544, 0.07625799626111984, 0.21829211711883545, -0.04628317058086395, 0.0923997089266777, 0.36629363894462585, 0.28515005111694336, 0.09202458709478378, -0.03479804843664169, 0.20943817496299744, -0.16328313946723938, 0.15064015984535217, 0.5692938566207886, 0.2252194583415985, -0.21879619359970093, -0.16825713217258453, -0.2295863777399063, 0.37271904945373535, 0.07174782454967499, -0.26780110597610474, 0.06641959398984909, 0.0441545695066452, 0.5

In [21]:
# compute toy_story vector norm for cosine similarity i.e L2 norm
toy_story_norm = np.linalg.norm(np.array(toy_story_vec, dtype=float))
toy_story_norm

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

np.float64(1.7576613303340416)

In [22]:
# build a list comprehension to compute dot product each movie latent feature vec * each toy_story vec for later execution
dot_movie_factors_toy_story = ' + '.join([
    f'features[{i}] * {float(toy_story_vec[i])}'    
    for i in range(len(toy_story_vec))
])

# build a list comprehension to compute the movie latent feature vector norm for later execution
movie_factors_norm = ' + '.join([
    f'features[{i}] * features[{i}]'
    for i in range(len(toy_story_vec))
])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
dot_movie_factors_toy_story

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

'features[0] * 0.3150031566619873 + features[1] * -0.21948567032814026 + features[2] * -0.1790868192911148 + features[3] * 0.21488389372825623 + features[4] * -0.47186562418937683 + features[5] * -0.08558786660432816 + features[6] * -0.09112725406885147 + features[7] * 0.072154700756073 + features[8] * 0.2768399715423584 + features[9] * 0.3999905586242676 + features[10] * -0.3012489676475525 + features[11] * 0.4461914896965027 + features[12] * 0.08083394169807434 + features[13] * 0.0019795196130871773 + features[14] * -0.20846426486968994 + features[15] * -0.0017408025451004505 + features[16] * 0.3032161593437195 + features[17] * 0.4313260018825531 + features[18] * -0.48546427488327026 + features[19] * 0.17604510486125946 + features[20] * 0.10194650292396545 + features[21] * 0.030576638877391815 + features[22] * 0.07829190045595169 + features[23] * 0.09839745610952377 + features[24] * 0.0017936719814315438 + features[25] * 0.040128812193870544 + features[26] * 0.07625799626111984 + fea

In [24]:
movie_factors_norm

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

'features[0] * features[0] + features[1] * features[1] + features[2] * features[2] + features[3] * features[3] + features[4] * features[4] + features[5] * features[5] + features[6] * features[6] + features[7] * features[7] + features[8] * features[8] + features[9] * features[9] + features[10] * features[10] + features[11] * features[11] + features[12] * features[12] + features[13] * features[13] + features[14] * features[14] + features[15] * features[15] + features[16] * features[16] + features[17] * features[17] + features[18] * features[18] + features[19] * features[19] + features[20] * features[20] + features[21] * features[21] + features[22] * features[22] + features[23] * features[23] + features[24] * features[24] + features[25] * features[25] + features[26] * features[26] + features[27] * features[27] + features[28] * features[28] + features[29] * features[29] + features[30] * features[30] + features[31] * features[31] + features[32] * features[32] + features[33] * features[33] +

In [25]:
# compute cosine similarities for all movies

cosine_similarities = (
    movie_factors.filter('id != 1')   # exclude Toy Story itself
      .select(
          F.col('id').alias('MovieID'),
          F.expr(f'({dot_movie_factors_toy_story}) / (sqrt({movie_factors_norm}) * {toy_story_norm})').alias('score')
      )
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [26]:
# get top 10 most similar movies

top_similar_movies = cosine_similarities.orderBy(F.col('score').desc()).limit(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [27]:
# get movie title (join)
toy_story_name = movies.filter('MovieID = 1').select('Title').first()[0]

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [28]:
top_similar_movies_with_names = top_similar_movies.join(movies, 'MovieID').select(
    F.lit(toy_story_name).alias('Movie Name'),
    F.col('Title').alias('Similar Movies'),
    F.col('score'))

# order by score descending
top_similar_movies_with_names = top_similar_movies_with_names.orderBy(F.col("score").desc())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [29]:
top_similar_movies_with_names.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------+--------------------+------------------+
|      Movie Name|      Similar Movies|             score|
+----------------+--------------------+------------------+
|Toy Story (1995)|  Toy Story 2 (1999)| 0.994736063152636|
|Toy Story (1995)|Bug's Life, A (1998)| 0.988812718013241|
|Toy Story (1995)| Finding Nemo (2003)|0.9692895329163022|
|Toy Story (1995)|Monsters, Inc. (2...|0.9691854262046992|
|Toy Story (1995)|Incredibles, The ...|0.9633633776642014|
|Toy Story (1995)|      Aladdin (1992)|0.9613345214505101|
|Toy Story (1995)|Ghosts of the Aby...|0.9601362761558054|
|Toy Story (1995)|       Tarzan (1999)|0.9579634324925023|
|Toy Story (1995)|         Antz (1998)|0.9568504612387276|
|Toy Story (1995)|Ace High (Quattro...|0.9551416161031842|
+----------------+--------------------+------------------+